# Paquetes

In [ ]:
# Análisis de datos 
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import KNNImputer

# Visualización
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Ruta y df

In [ ]:
# Función para encontrar el archivo CSV
def encontrar_csv(root=Path.cwd()):
    ruta_csv = root / 'data' / 'raw' / 'dataset_Regresión.csv'
    if ruta_csv.exists():
        return ruta_csv
    elif root != root.parent:
        root = root.parent
        return encontrar_csv(root)
    else:
        raise FileNotFoundError("No se encontró el archivo dataset_Regresión.csv en el directorio actual ni en los directorios superiores.")

ruta_csv = encontrar_csv()
print("Usando ruta:", ruta_csv)

df = pd.read_csv(ruta_csv)

df.columns = df.columns.str.strip()

df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]

print(f"Dimensiones: {df.shape}")

df.head()



# 1. Corrección de datos inconsistentes e imputación

# Corrección de datos

In [ ]:
columnas=list(df.columns)

# Valores inconsistentes o erróneos en las columnas

valor1=df[columnas[25]].unique()[2]
valor2=df[columnas[26]].unique()[135]
valor3=df[columnas[28]].unique()[161]

# Reemplazo

df[columnas[25]] = df[columnas[25]].replace(valor1, "No")
df[columnas[26]] = df[columnas[26]].replace(valor2, 1.42)
df[columnas[28]] = df[columnas[28]].replace(valor3, 3.1)

# Análisis detallado en src/Regresión/Corrección_dataset_Regresión.ipynb

# Datos faltantes

In [ ]:
df.isna().sum()

# Imputación

In [ ]:
# Método de Media
df_media=df.fillna(df.mean(numeric_only=True))


# Método de KNN
# 1. Aísla solo las columnas numéricas
cols_numericas = df.select_dtypes(include='number').columns

# 2. Aplica KNN solo a esas columnas
imputer_knn = KNNImputer(n_neighbors=5)
df_numeric_imputed = pd.DataFrame(
    imputer_knn.fit_transform(df[cols_numericas]),
    columns=cols_numericas,
    index=df.index
)

# 3. Copia el df original y reemplaza solo las columnas numéricas
df_knn = df.copy()
df_knn[cols_numericas] = df_numeric_imputed

# Comparación de resultados

print("\n ORIGINAL:")
print(df.describe())
print("="*120)
print("\n MEDIA:")
print(df_media.describe())
print("="*120)
print("\n KNN:")
print(df_knn.describe())

# Elejir método de imputación

In [ ]:
copia=df.copy()

# Elección por ahora
#df=copia
df=df_media
#df=df_knn


def elejir_imputacion(df):
    print("Elige el método de imputación:")
    print("1. Original (sin imputar)")
    print("2. Media")
    print("3. KNN")
    eleccion = input("Ingresa el número correspondiente a tu elección: ")
    while eleccion not in ['1', '2', '3']:
        print("Opción no válida. Por favor, elige 1, 2 o 3.")
        eleccion = input("Ingresa el número correspondiente a tu elección: ")
    if eleccion==1:
        df=copia
    elif eleccion==2:
        df=df_media
    else:
        df=df_knn
    print("Imputación seleccionada correctamente")
    return df.isna().sum()

# 2. Tipos de variables

In [ ]:
def indentificar_tipos_de_variables(df,umbral_categoricas=8):
    """ 
    umbral_categoricas: número máximo de valores únicos para considerar una variable numérica como categórica
    """
    categoricas = []
    numericas = []
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            if df[col].nunique() <= umbral_categoricas and df[col].nunique()<len(df)*0.05:
                categoricas.append(col)
            else:
                numericas.append(col)
        else:
            categoricas.append(col)
    
    return categoricas, numericas

# Clasificar variables
categoricas, numericas = indentificar_tipos_de_variables(df)

# Mostrar resultados
print("Variables numericas:")
for var in numericas:
    print(f" - {var} -> {df[var].nunique()} valores únicos, tipo: {df[var].dtype}")

print("\nVariables categóricas:")
for var in categoricas:
    print(f" - {var} -> {df[var].nunique()} valores únicos, tipo: {df[var].dtype}")

In [ ]:
# Valores unicos por variable

col=list(df.columns)

for i,var in enumerate(col):
    print("="*120)
    print(f"{i}.Valores de {var}: {df[var].unique()}")

# Variables a estudiar

In [ ]:
cat,num=indentificar_tipos_de_variables(df)

In [ ]:
Valores_a_quitar_cat=['Program','What are the skills do you have ?',
                  'What is you interested area?',]

categoricas=[var for var in cat if var not in Valores_a_quitar_cat]

Variables_mas_importante_categoricas = categoricas

In [ ]:
Variables_mas_importante_numericas = num

# 3. Análisis de variables categóricas

## Plots ##

In [ ]:
for var in Variables_mas_importante_categoricas:
    plt.figure(figsize=(8, 4))
    sns.countplot(x=var, data=df.dropna())
    plt.title(f"Plot de {var}")
    plt.xticks(rotation=45)
    plt.show()

# Correlación

In [ ]:
# Correlación entre CGPA y variables categóricas
from sklearn.preprocessing import LabelEncoder

target = "What is your current CGPA?"

# Crear copia del dataframe para no alterar el original
df_cat_corr = df[Variables_mas_importante_categoricas + [target]].copy()

# Eliminar filas con valores faltantes
df_cat_corr = df_cat_corr.dropna()

# Codificar variables categóricas
le_dict = {}
for col in Variables_mas_importante_categoricas:
    le = LabelEncoder()
    df_cat_corr[col + '_encoded'] = le.fit_transform(df_cat_corr[col])
    le_dict[col] = le

# Seleccionar solo columnas codificadas y el target
df_encoded = df_cat_corr[[col + '_encoded' for col in Variables_mas_importante_categoricas] + [target]]
df_encoded.columns = [col.replace('_encoded', '') for col in df_encoded.columns]

# Calcular correlación con CGPA
corr_cat = df_encoded.corr()[target].drop(target).sort_values(ascending=False)

print("Correlación de variables categóricas con CGPA:")
print("=" * 70)
for var, corr_val in corr_cat.items():
    print(f"{var:60} {corr_val:7.4f}")

# Grafico
plt.figure(figsize=(12, 6))
corr_cat.plot(kind='barh', color='lightcoral', edgecolor='black')
plt.title('Correlación de Variables Categóricas con CGPA', fontsize=16, fontweight='bold')
plt.xlabel('Correlación', fontsize=12)
plt.ylabel('Variables', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()


## BoxPlots

In [ ]:
# Análisis de relación entre variables - Scatter plots

target = "What is your current CGPA?"

# Gráficos de boxplot para visualizar la relación
print("\nBoxplots de CGPA por categoría:")
for var in Variables_mas_importante_categoricas:
    try:
        temp_df = df[[var, target]].dropna()
        
        if len(temp_df) > 0:
            plt.figure(figsize=(12, 7))
            temp_df.boxplot(column=target, by=var, figsize=(12, 7))
            plt.suptitle('')
            plt.xlabel(var, fontsize=12, fontweight='bold')
            plt.ylabel(target, fontsize=12, fontweight='bold')
            plt.title(f'Distribución de CGPA por {var}', fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()
        else:
            print(f"  Sin datos válidos para {var}")
    except Exception as e:
        print(f"  Error en {var}: {e}")

# 4. Análisis de variables numericas

# Análisis descriptivo

In [ ]:
def Describir_numericas(df, Variables_mas_importante):
    numericas = indentificar_tipos_de_variables(df)[1]
    for var in Variables_mas_importante_numericas:
        print(f"===================================================================================")
        print(f"Análisis {var}")
        print(f"===================================================================================")
        print(f"-Promedio: {df[var].mean()}")
        print(f"-Mediana: {df[var].median()}")
        print(f"-Varianza: {df[var].var()}")
        print(f"-Cuantiles: {df[var].quantile([0.25,0.5,0.75])}")

Describir_numericas(df, Variables_mas_importante_numericas )

# Valores atípicos

In [ ]:
# Dedectar valor atípicos
def detectar_outliers(df, variable):
    Q1 = df[variable].quantile(0.25)
    Q3 = df[variable].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[variable] < lower_bound) | (df[variable] > upper_bound)]
    return outliers

for Variable in Variables_mas_importante_numericas:
    print(f"===================================================================================")
    print(f"Análisis de outliers para {Variable}")
    print(f"===================================================================================")
    outliers = detectar_outliers(df, Variable)
    print(f"-Número de outliers: {len(outliers)}")
    print(f"-Valores atípicos:\n{outliers[Variable].tolist()}")

# Histogramas


In [ ]:
def crear_histogramas_sin_outliers(df, columna, bins=None, color="skyblue", titulo=None, clip_outliers=True):
    data = df[columna].dropna()

    # Calcular límites de outliers con IQR
    q75, q25 = np.percentile(data, [75, 25])
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    data_sin_outliers = data[(data >= lower_bound) & (data <= upper_bound)]
    n_outliers = len(data) - len(data_sin_outliers)

    if clip_outliers and len(data_sin_outliers) >= max(3, len(data) * 0.5):
        data_plot = data_sin_outliers
        rango_text = f"(sin outliers: {lower_bound:.2f} a {upper_bound:.2f})"
    else:
        data_plot = data
        rango_text = "(incluye todos los valores)"

    # Calcular bins óptimos si no se especifican
    if bins is None:
        if data_plot.nunique() <= 20:
            bins = data_plot.nunique()
        else:
            q75p, q25p = np.percentile(data_plot, [75, 25])
            iqrp = q75p - q25p
            if iqrp > 0:
                bin_width = 2 * iqrp / (len(data_plot) ** (1/3))
                bins = int(np.ceil((data_plot.max() - data_plot.min()) / bin_width))
                bins = max(15, min(bins, 40))
            else:
                bins = 20

    # Crear figura y eje
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_facecolor('#fafafa')
    fig.patch.set_facecolor('white')

    # Histograma
    ax.hist(data_plot, bins=bins, color=color, edgecolor='black', alpha=0.85)

    # Títulos
    if titulo is None:
        titulo = f"Distribución de {columna} {rango_text}"
    ax.set_title(titulo, fontsize=16, fontweight='bold')
    ax.set_xlabel(columna, fontsize=12)
    ax.set_ylabel('Frecuencia', fontsize=12)

    # Estadística descriptiva
    Mediana = data.median()
    Promedio = data.mean()
    Varianza = data.var()

    # Líneas de estadística descriptiva
    ax.axvline(Mediana, color='red', linestyle='dashed', linewidth=1.5, label=f'Mediana: {Mediana:.2f}')
    ax.axvline(Promedio, color='green', linestyle='dashed', linewidth=1.5, label=f'Promedio: {Promedio:.2f}')
    ax.legend(loc='upper left', frameon=True, framealpha=0.95, fontsize=10)

    # Información estadística en el gráfico
    stats_text = (
        f'n = {len(data)}\n'
        f'Media = {Promedio:.2f}\n'
        f'Mediana = {Mediana:.2f}\n'
        f'Varianza = {Varianza:.2f}\n'
        f'Rango datos = {data.min():.2f} - {data.max():.2f}\n'
        f'Valores usados = {len(data_plot)} ({n_outliers} outliers excluidos)'
    )
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='#cccccc'), fontsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
# Verificar las columnas y generar histogramas para cada variable importante
# Normalizar nombres de columnas y la lista de variables importantes
if not df.columns.equals(df.columns.str.strip()):
    df.columns = df.columns.str.strip()

variables_norm = [col.strip() for col in Variables_mas_importante_numericas]
cols_norm = [col.strip() for col in df.columns]

missing = [col for col in variables_norm if col not in cols_norm]
if missing:
    import difflib
    print("Columnas no encontradas en el DataFrame:")
    for col in missing:
        print(f" - {repr(col)}")
        matches = difflib.get_close_matches(col, cols_norm, n=3, cutoff=0.6)
        if matches:
            print(f"   Coincidencias cercanas: {matches}")
    print("\nAsegúrate de que los nombres coincidan exactamente en la lista y en el DataFrame.")
else:
    print("Todas las columnas están presentes. Generando histogramas...")
    for col in variables_norm:
        print(f"\nGenerando histograma para: {col}")
        crear_histogramas_sin_outliers(df, col)

In [ ]:
def crear_histogramas_con_outliers(df, columna, bins=None, color="skyblue", titulo=None):
    data = df[columna].dropna()

    # Calcular bins óptimos si no se especifican
    if bins is None:
        if data.nunique() <= 20:
            bins = data.nunique()
        else:
            q75, q25 = np.percentile(data, [75, 25])
            iqr = q75 - q25
            if iqr > 0:
                bin_width = 2 * iqr / (len(data) ** (1/3))
                bins = int(np.ceil((data.max() - data.min()) / bin_width))
                bins = max(15, min(bins, 40))
            else:
                bins = 20

    # Crear figura y eje
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_facecolor('#fafafa')
    fig.patch.set_facecolor('white')

    # Histograma
    ax.hist(data, bins=bins, color=color, edgecolor='black', alpha=0.85)

    # Títulos
    if titulo is None:
        titulo = f"Distribución de {columna}"
    ax.set_title(titulo, fontsize=16, fontweight='bold')
    ax.set_xlabel(columna, fontsize=12)
    ax.set_ylabel('Frecuencia', fontsize=12)

    # Estadística descriptiva
    Mediana = data.median()
    Promedio = data.mean()
    Varianza = data.var()

    # Líneas de estadística descriptiva
    ax.axvline(Mediana, color='red', linestyle='dashed', linewidth=1.5, label=f'Mediana: {Mediana:.2f}')
    ax.axvline(Promedio, color='green', linestyle='dashed', linewidth=1.5, label=f'Promedio: {Promedio:.2f}')
    ax.legend(loc='upper left', frameon=True, framealpha=0.95, fontsize=10)

    # Información estadística en el gráfico
    stats_text = f'n = {len(data)}\nMedia = {Promedio:.2f}\nMediana = {Mediana:.2f}\nVarianza = {Varianza:.2f}\nRango: {data.min():.2f} - {data.max():.2f}'
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='#cccccc'), fontsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
# Verificar las columnas y generar histogramas para cada variable importante
# Normalizar nombres de columnas y la lista de variables importantes
if not df.columns.equals(df.columns.str.strip()):
    df.columns = df.columns.str.strip()

variables_norm = [col.strip() for col in Variables_mas_importante_numericas]
cols_norm = [col.strip() for col in df.columns]

missing = [col for col in variables_norm if col not in cols_norm]
if missing:
    import difflib
    print("Columnas no encontradas en el DataFrame:")
    for col in missing:
        print(f" - {repr(col)}")
        matches = difflib.get_close_matches(col, cols_norm, n=3, cutoff=0.6)
        if matches:
            print(f"   Coincidencias cercanas: {matches}")
    print("\nAsegúrate de que los nombres coincidan exactamente en la lista y en el DataFrame.")
else:
    print("Todas las columnas están presentes. Generando histogramas...")
    for col in variables_norm:
        print(f"\nGenerando histograma para: {col}")
        crear_histogramas_con_outliers(df, col)

# Correlaciones

In [ ]:
# Calcular correlación con Spearman entre variables numéricas

# Variable objetivo
target="What is your current CGPA?"

# Seleccionar solo columnas numéricas
df_numeric = df.select_dtypes(include=[np.number]).dropna()

# Correlación con la variable objetivo
corr_target = df_numeric.corr(method='spearman')[target].drop(target).sort_values(ascending=False)

# Grafico
corr_target.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title(f'Correlación de variables con {target}', fontsize=16, fontweight='bold')
plt.ylabel('Correlación', fontsize=12)
plt.xlabel('Variables', fontsize=12)
plt.axvline(x=0, color='black',linestyle='--', linewidth=0.8) #Linea en cero
plt.show()

In [ ]:
# Calcular correlación con Kendall entre variables numéricas

# Variable objetivo
target="What is your current CGPA?"

# Seleccionar solo columnas numéricas
df_numeric = df.select_dtypes(include=[np.number]).dropna()

# Correlación con la variable objetivo
corr_target = df_numeric.corr(method='kendall')[target].drop(target).sort_values(ascending=False)

# Grafico
corr_target.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title(f'Correlación de variables con {target}', fontsize=16, fontweight='bold')
plt.ylabel('Correlación', fontsize=12)
plt.xlabel('Variables', fontsize=12)
plt.axvline(x=0, color='black',linestyle='--', linewidth=0.8) #Linea en cero
plt.show()

In [ ]:
# Calcular correlación con Pearson entre variables numéricas

# Variable objetivo
target="What is your current CGPA?"

# Seleccionar solo columnas numéricas
df_numeric = df.select_dtypes(include=[np.number]).dropna()

# Correlación con la variable objetivo
corr_target = df_numeric.corr(method='pearson')[target].drop(target).sort_values(ascending=False)

# Grafico
corr_target.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title(f'Correlación de variables con {target}', fontsize=16, fontweight='bold')
plt.ylabel('Correlación', fontsize=12)
plt.xlabel('Variables', fontsize=12)
plt.axvline(x=0, color='black',linestyle='--', linewidth=0.8) #Linea en cero
plt.show()

# Scatter plots

In [ ]:
target = "What is your current CGPA?"

# Gráficos de Scatter para visualizar la relación
print("\nScatter de CGPA por cada variable numérica:")
for var in Variables_mas_importante_numericas:
    if var != target:  # No graficar la variable contra sí misma
        # Eliminar filas con NaN en ambas columnas
        temp_df = df[[var, target]].dropna()
        
        if len(temp_df) > 0:
            plt.figure(figsize=(12, 10))
            plt.scatter(temp_df[var], temp_df[target], alpha=0.6, edgecolors='k', s=70, color='steelblue')
            plt.title(f'Relación entre {var} y CGPA', fontsize=14, fontweight='bold')
            plt.xlabel(var, fontsize=12)
            plt.ylabel(target, fontsize=12)
            plt.grid(True, alpha=1)
            plt.tight_layout()
            plt.show()
        else:
            print(f"  Sin datos válidos para {var}")

# 5. Pruebas de asociación y dependencia

## Prueba chi-cuadrado para variables categoricas

In [ ]:
from scipy.stats import chi2_contingency

def prueba_chi(cate=Variables_mas_importante_categoricas, alpha=0.05):

    # Interfaz usuario:
    print("\n PRUEBA CHI-CUADRADO")
    print("\n Elija un número para la información que proporcionara la prueba:")
    print("\n 1. Todo")
    print("\n 2. Con Dependencia")
    print("\n 3. Con Independencia")

    resultado=input("\n Elija un número:")

    if resultado=='1':
        print("\n VARIABLES CON DEPENDENCIA E INDEPENDENCIA")
        for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                # Tabla de contingencia
                tabla=pd.crosstab(df[var1],df[var2])

                #Prueba Chi-cuadrado
                chi2,p,dof,expected=chi2_contingency(tabla)

                print("="*150)
                print(f"\n {var1} -- VS -- {var2} ")
                print(f"\n Estadistico: {chi2:.4f}, p-valor: {p:.4f}")
                if p<alpha:
                    print(f"\n Existe evidencia de asosiación entre las variables")
                else:
                    print(f"\n No hay evidencia sufiente de asosiación entre las variables")

    if resultado=='2':
         print("\n VARIABLES CON DEPENDENCIA")
         for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                # Tabla de contingencia
                tabla=pd.crosstab(df[var1],df[var2])

                #Prueba Chi-cuadrado
                chi2,p,dof,expected=chi2_contingency(tabla)

                if p<alpha:
                    print("="*150)
                    print(f"\n {var1} -- VS -- {var2} ")
                    print(f"\n Estadistico: {chi2:.4f}, p-valor: {p:.4f}")

    if resultado=='3':
         print("\n VARIABLES CON INDEPENDENCIA")
         for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                # Tabla de contingencia
                tabla=pd.crosstab(df[var1],df[var2])

                #Prueba Chi-cuadrado
                chi2,p,dof,expected=chi2_contingency(tabla)

                if p>=alpha:
                    print("="*150)
                    print(f"\n {var1} -- VS -- {var2} ")
                    print(f"\n Estadistico: {chi2:.4f}, p-valor: {p:.4f}")

        
prueba_chi()


 Elija un número para la información que proporcionara la prueba:

 1. Todo

 2. Con Dependencia

 3. Con Independencia


## Prueba de Spearman para variables númericas

In [87]:
from scipy.stats import spearmanr

def prueba_spearmanr(cate=Variables_mas_importante_numericas, alpha=0.05):

    # Interfaz usuario:
    print("\n PRUEBA SPEARMANR")
    print("\n Elija un número para la información que proporcionara la prueba:")
    print("\n 1. Todo")
    print("\n 2. Con Asociación monótona")
    print("\n 3. Sin asociación monótona ")

    resultado=input("\n Elija un número:")

    if resultado=='1':
        print("\n VARIABLES CON Y SIN ASOCIACIÓN MONÓTONA")
        for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                #Prueba Spearmanr
                coef,p=spearmanr(df[var1],df[var2])

                fuerza=""
                if abs(coef)<0.3:
                    fuerza="Débil"
                elif abs(coef)< 0.7:
                    fuerza="Moderada"
                else:
                    fuerza="Fuerte"

                direccion="Positiva" if coef>0 else "Negativa"

                print("="*150)
                print(f"\n {var1} -- VS -- {var2} ")
                print(f"\n Coeficiente: {round(abs(coef),5)}, Fuerza: {fuerza}, Dirección: {direccion}, p-valor: {p:.4f}")
                if p<alpha:
                    print(f"\n Existe evidencia de asosiación entre las variables")
                else:
                    print(f"\n No hay evidencia sufiente de asosiación entre las variables")

    if resultado=='2':
         print("\n VARIABLES CON ASOCIACIÓN MONÓTONA")
         for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                #Prueba Spearmanr
                coef,p=spearmanr(df[var1],df[var2])

                fuerza=""
                if abs(coef)<0.3:
                    fuerza="Débil"
                elif abs(coef)< 0.7:
                    fuerza="Moderada"
                else:
                    fuerza="Fuerte"

                direccion="Positiva" if coef>0 else "Negativa"

                if p<alpha:
                    print("="*150)
                    print(f"\n {var1} -- VS -- {var2} ")
                    print(f"\n Coeficiente: {round(abs(coef),5)}, Fuerza: {fuerza}, Dirección: {direccion}, p-valor: {p:.4f}")

    if resultado=='3':
         print("\n VARIABLES SIN ASOCIACIÓN MONÓTONA")
         for var1 in cate:
            for var2 in cate:
                if var1==var2:
                    break

                #Prueba Spearmanr
                coef,p=spearmanr(df[var1],df[var2])

                fuerza=""
                if abs(coef)<0.3:
                    fuerza="Débil"
                elif abs(coef)< 0.7:
                    fuerza="Moderada"
                else:
                    fuerza="Fuerte"

                direccion="Positiva" if coef>0 else "Negativa"

                if p>=alpha:
                    print("="*150)
                    print(f"\n {var1} -- VS -- {var2} ")
                    print(f"\n Coeficiente: {round(abs(coef),5)}, Fuerza: {fuerza}, Dirección: {direccion}, p-valor: {p:.4f}")

        
prueba_spearmanr()


 PRUEBA SPEARMANR

 Elija un número para la información que proporcionara la prueba:

 1. Todo

 2. Con Asociación monótona

 3. Sin asociación monótona 
